In [1]:
import geocat
import datetime
import pandas as pd
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from os import listdir
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker

In [ ]:
# 常用函数

## 从NC文件时间转PD时间
def cftime2pdtime(cf):
    return pd.to_datetime(datetime.datetime(cf.year,cf.month,cf.day,cf.hour))

## 获取文件夹里所有NC文件
def getfiles(filed):
    files=listdir(filed)
    files.sort()
    files=files[0:]
    files_n=[filed+'/'+i for i in files]
    return files_n

from datetime import date

def runavg(ts, w):
    '''
    Performs a running average of an input time series using uniform window
    of width w. This function assumes that the input time series is periodic.
    Inputs:
      ts            Time series [1D numpy array]
      w             Integer length (must be odd) of running average window
    Outputs:
      ts_smooth     Smoothed time series
    Written by Eric Oliver, Institue for Marine and Antarctic Studies, University of Tasmania, Feb-Mar 2015
    '''
    # Original length of ts
    N = len(ts)
    # make ts three-fold periodic
    ts = np.append(ts, np.append(ts, ts))
    # smooth by convolution with a window of equal weights
    ts_smooth = np.convolve(ts, np.ones(w)/w, mode='same')
    # Only output central section, of length equal to the original length of ts
    ts = ts_smooth[N:2*N]

    return ts

def Clim(t, data, climatologyPeriod=[None,None],  windowHalfWidth=5, smoothPercentile=True, smoothPercentileWidth=31,alternateClimatology=False, Ly=False):
    T = len(t)
    year = np.zeros((T))
    month = np.zeros((T))
    day = np.zeros((T))
    doy = np.zeros((T))
    for i in range(T):
        year[i] = date.fromordinal(t[i]).year
        month[i] = date.fromordinal(t[i]).month
        day[i] = date.fromordinal(t[i]).day
    # Leap-year baseline for defining day-of-year values
    year_leapYear = 2012 # This year was a leap-year and therefore doy in range of 1 to 366
    t_leapYear = np.arange(date(year_leapYear, 1, 1).toordinal(),date(year_leapYear, 12, 31).toordinal()+1)
    dates_leapYear = [date.fromordinal(tt.astype(int)) for tt in t_leapYear]
    month_leapYear = np.zeros((len(t_leapYear)))
    day_leapYear = np.zeros((len(t_leapYear)))
    doy_leapYear = np.zeros((len(t_leapYear)))
    for tt in range(len(t_leapYear)):
        month_leapYear[tt] = date.fromordinal(t_leapYear[tt]).month
        day_leapYear[tt] = date.fromordinal(t_leapYear[tt]).day
        doy_leapYear[tt] = t_leapYear[tt] - date(date.fromordinal(t_leapYear[tt]).year,1,1).toordinal() + 1
    # Calculate day-of-year values
    for tt in range(T):
        doy[tt] = doy_leapYear[(month_leapYear == month[tt]) * (day_leapYear == day[tt])]

    # Constants (doy values for Feb-28 and Feb-29) for handling leap-years
    feb28 = 59
    feb29 = 60

    # Set climatology period, if unset use full range of available data
    if (climatologyPeriod[0] is None) or (climatologyPeriod[1] is None):
        climatologyPeriod[0] = year[0]
        climatologyPeriod[1] = year[-1]

    #
    # Calculate threshold and seasonal climatology (varying with day-of-year)
    #

    # if alternate temperature time series is supplied for the calculation of the climatology
    if alternateClimatology:
        tClim = alternateClimatology[0]
        tempClim = alternateClimatology[1]
        TClim = len(tClim)
        yearClim = np.zeros((TClim))
        monthClim = np.zeros((TClim))
        dayClim = np.zeros((TClim))
        doyClim = np.zeros((TClim))
        for i in range(TClim):
            yearClim[i] = date.fromordinal(tClim[i]).year
            monthClim[i] = date.fromordinal(tClim[i]).month
            dayClim[i] = date.fromordinal(tClim[i]).day
            doyClim[i] = doy_leapYear[(month_leapYear == monthClim[i]) * (day_leapYear == dayClim[i])]
    else:
        tempClim = data.copy()
        TClim = np.array([T]).copy()[0]
        yearClim = year.copy()
        monthClim = month.copy()
        dayClim = day.copy()
        doyClim = doy.copy()
    
    
    
    lenClimYear = 366
    # Start and end indices
    clim_start = np.where(yearClim == climatologyPeriod[0])[0][0]
    clim_end = np.where(yearClim == climatologyPeriod[1])[0][-1]
    # Inialize arrays
    seas_climYear = np.NaN*np.zeros(lenClimYear)
    clim = {}
    clim['seas'] = np.NaN*np.zeros(TClim)
    # Loop over all day-of-year values, and calculate threshold and seasonal climatology across years
    for d in range(1,lenClimYear+1):
        # Special case for Feb 29
        if d == feb29:
            continue
        # find all indices for each day of the year +/- windowHalfWidth and from them calculate the threshold
        tt0 = np.where(doyClim[clim_start:clim_end+1] == d)[0] 
        # If this doy value does not exist (i.e. in 360-day calendars) then skip it
        if len(tt0) == 0:
            continue
        tt = np.array([])
        for w in range(-windowHalfWidth, windowHalfWidth+1):
            tt = np.append(tt, clim_start+tt0 + w)
        tt = tt[tt>=0] # Reject indices "before" the first element
        tt = tt[tt<TClim] # Reject indices "after" the last element
        seas_climYear[d-1] = np.nanmean(tempClim[tt.astype(int)])
    # Special case for Feb 29
    seas_climYear[feb29-1] = 0.5*seas_climYear[feb29-2] + 0.5*seas_climYear[feb29]

    #return seas_climYear
    # Smooth if desired
    if smoothPercentile:
        # If the length of year is < 365/366 (e.g. a 360 day year from a Climate Model)
        if Ly:

            valid = ~np.isnan(seas_climYear)
            seas_climYear[valid] = runavg(seas_climYear[valid], smoothPercentileWidth)
        # >= 365-day year
        else:

            seas_climYear = runavg(seas_climYear, smoothPercentileWidth)

    #print(doy)
    #clim['seas'] = seas_climYear[doy.astype(int)-1]

    # Save vector indicating which points in temp are missing values
    #clim['missing'] = np.isnan(data)
    # Set all remaining missing temp values equal to the climatology
    #data[np.isnan(data)] = clim['seas'][np.isnan(data)]

    return seas_climYear[doy.astype(int)-1]
# fl=getfiles("/lustre/home/yuhanxue/data/oisst/V2.1")

In [ ]:
ssts=np.load('/lustre/home/yuhanxue/data/oisstv2r01_ssts.npy')
times=np.load('/lustre/home/yuhanxue/data/oisstv2r01_times.npy',allow_pickle=True)
das=nc.Dataset("/lustre/home/yuhanxue/data/oisst/V2.1/oisst-avhrr-v02r01.20200101.nc")
lat=np.array(das.variables['lat'])
lon=np.array(das.variables['lon'])
lonind=(lon>=80)&(lon<=300)
latind=(lat>=-40)&(lat<=65)

t= np.arange(date(1981,9,1).toordinal(),date(2022,10,13).toordinal()+1)# For ClimCal
ts=pd.date_range('2020-01-01','2022-01-01',freq='1d')
timeind=(times>=ts[0])&(times<ts[-1])

In [ ]:
plt.figure(figsize=(40,10),dpi=300)
plt.subplot(2,1,1)
plt.title('Climatology')
plt.plot(times[timeind],ssts[360,360,timeind],label='sst')
plt.plot(times[timeind],Clim(t,ssts[360,360,:],climatologyPeriod=[1982,2021],smoothPercentile=False)[timeind],label='No Smooth')
plt.plot(times[timeind],Clim(t,ssts[360,360,:],climatologyPeriod=[1982,2021],smoothPercentile=True)[timeind],label='Smooth')
plt.xlim(times[timeind][0],times[timeind][-1])
plt.legend()
plt.subplot(2,1,2)
plt.title('Anomaly')
plt.plot(times[timeind],ssts[360,360,timeind]-Clim(t,ssts[360,360,:],climatologyPeriod=[1982,2021],smoothPercentile=False)[timeind],label='No Smooth')
plt.plot(times[timeind],ssts[360,360,timeind]-Clim(t,ssts[360,360,:],climatologyPeriod=[1982,2021],smoothPercentile=True)[timeind],label='Smooth')
plt.xlim(times[timeind][0],times[timeind][-1])
plt.legend()
plt.suptitle('ClimatologyPeriod[1982,2021]')


In [ ]:
def clim_core(dat):
    global t
    if np.sum(np.isnan(dat))>=100:
        a=np.zeros(shape=t.shape)
        a[:]=np.nan
        return a
    else:
        return Clim(t,dat,climatologyPeriod=[1982,2021],smoothPercentile=True)
def list_map_clim(dat):
    pool = ProcessPoolExecutor(max_workers=24)
    ans=np.array(list(map(clim_core,dat)))
    del pool
    print('36 over')
    return ans
def Clim_allinone(dat):
    pool = ProcessPoolExecutor(max_workers=12)
    ans=np.array(list(pool.map(list_map_clim,dat)))
    del pool
    return ans
SSTC=Clim_allinone(ssts)
SSTA=ssts-SSTC

In [ ]:
plt.figure(figsize=[20,10])
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
Lon,Lat=np.meshgrid(lon[lonind],lat[latind])
c=ax.contourf(Lon,Lat,ssts[:,lonind,0][latind],transform=ccrs.PlateCarree(central_longitude=0))
ax.xaxis.set_major_formatter(LongitudeFormatter())#刻度格式转换为经纬度样式 
ax.yaxis.set_major_formatter(LatitudeFormatter())
ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))#刻度格式转换为经纬度样式 
ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.tick_params(axis='both',which='major',labelsize=15,direction='out',length=5,width=0.3,pad=0.2,top=True,right=True)
ax.coastlines() 
ax.spines['geo'].set_linewidth(0.5)#调节边框粗细
ax.set_xticks(np.arange(int(np.nanmin(Lon)),int(np.nanmax(Lon)+1),15)-180)
ax.set_yticks(np.arange(int(np.nanmin(Lat)-1),int(np.nanmax(Lat)+1),10))
#plt.grid()
plt.colorbar(c,orientation='horizontal',label='SST (℃)',shrink=0.7)
plt.show()


In [ ]:
np.save('ssta.npy',SSTA)

In [ ]:
Lon,Lat=np.meshgrid(lon[lonind],lat[latind])
def draw(i):
    global SSTA,times,Lon,Lat,ts
    plt.cla()
    plt.ioff()
    ig, ax = plt.subplots(figsize=(20,10),dpi=300,subplot_kw=dict(facecolor='white', projection=ccrs.PlateCarree(central_longitude=180)))
    ax.set_title(f"SSTA {ts[i]}",size=25)
    c=ax.contourf(Lon,Lat,SSTA[:,:,times==ts[i]][:,:,0],cmap='RdBu_r',levels=np.arange(-4,4.1,0.1),transform=ccrs.PlateCarree(central_longitude=0))#
    ax.contourf(Lon,Lat,SSTA[:,:,times==ts[i]][:,:,0],colors=['#0d3f76'],levels=[-99,-4],transform=ccrs.PlateCarree(central_longitude=0))
    ax.contourf(Lon,Lat,SSTA[:,:,times==ts[i]][:,:,0],colors=['#7c0722'],levels=[4,99],transform=ccrs.PlateCarree(central_longitude=0))
    co=ax.contour(Lon,Lat,SSTA[:,:,times==ts[i]][:,:,0],levels=[-2,2],colors='k',linewidths=0.3,transform=ccrs.PlateCarree(central_longitude=0))
    ax.clabel(co, inline=True, fontsize=10)
    ax.xaxis.set_major_formatter(LongitudeFormatter())#刻度格式转换为经纬度样式 
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))#刻度格式转换为经纬度样式 
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
    ax.tick_params(axis='both',which='major',labelsize=15,direction='out',length=5,width=0.3,pad=0.2,top=True,right=True)
    ax.coastlines() 
    ax.spines['geo'].set_linewidth(0.5)#调节边框粗细
    ax.set_xticks(np.arange(int(np.nanmin(Lon)),int(np.nanmax(Lon)+1),15)-180)
    ax.set_yticks(np.arange(int(np.nanmin(Lat)-1),int(np.nanmax(Lat)+1),10))
    plt.colorbar(c,orientation='horizontal',label='SSTA (℃)',shrink=0.7)
    plt.savefig(f'{i}.png')
    plt.close()
#draw(0)
pool = ProcessPoolExecutor(max_workers=72)
list(pool.map(draw,np.arange(ts.shape[0])))
del pool

In [ ]:
# zip images.zip *.zip
# ffmpeg -r 4 -i %03d.png -c:v libx264 -preset ultrafast -threads 72 output.mp4